# PitWall AI: reproduce all the results (top to bottom)

ESSEC AIDAMS · Business Data Science · Group project (group F1).
This notebook re-runs the whole project from the repository: FIA data → anonymised datasets → index → RAG
evaluation → classifier fine-tuning and baselines → figures. Every number in the report comes from these cells.

**Before running:** `Runtime → Change runtime type → T4 GPU` (CPU works too, training is just slower), and add your
Groq key in the Colab **Secrets** panel (🔑 on the left) under the name `GROQ_API_KEY`.

In [ ]:
REPO_URL = "https://github.com/Coachbet/RAG_FOR_F1"
!git clone -q $REPO_URL pitwall || (cd pitwall && git pull -q)
%cd pitwall
!pip -q install -e . 2>&1 | tail -1

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
!python -c "from pitwall.config import settings as s; print(s.llm_provider, s.llm_model, s.embedding_model, s.retrieval_mode, 'MIN_SCORE', s.min_score)"

## 1. Data
The FIA documents are not redistributed: they are downloaded from fia.com (public pages, one request every 10 s
as robots.txt asks, so the first run of a full season takes 1 to 2 hours; everything is cached in `data/raw/fia/`).
Then `prepare_data.py` keeps the on-track incidents, anonymises the drivers, labels, splits and builds the RAG
documents. The human label corrections are committed in `data/ft/label_overrides.csv`.

In [ ]:
!python scripts/fetch_fia.py
!python scripts/prepare_data.py
import json, pandas as pd
print(json.dumps(json.load(open("data/processed/data_report.json")), indent=1))
pd.read_csv("data/processed/sources.csv").category.value_counts()

## 2. Index (chunk → embed → FAISS)

In [ ]:
!python scripts/build_index.py

## 3. Try a few questions (including one with no answer in the documents)

In [ ]:
!python scripts/ask.py "What must an overtaking car achieve on the inside of a corner?"
!python scripts/ask.py "What is the 2026 cost cap for teams?"

## 4. RAG evaluation
1. Calibrate the refusal threshold on the **DEV** questions only.
2. Retrieval metrics (dense vs hybrid) on **TEST**.
3. Full evaluation on **TEST**, with the "LLM without RAG" baseline.

In [ ]:
!python scripts/run_eval.py --sweep
# Copy the suggested value here, then it is used by every following cell:
os.environ["MIN_SCORE"] = "0.80"   # TODO: value printed above

In [ ]:
!python scripts/run_eval.py --retrieval-only
!python scripts/run_eval.py --baseline

In [ ]:
import glob
rag = pd.read_csv(sorted(glob.glob("eval/results/*_rag_test_*.csv"))[-1])
rag[["qid", "question", "answerable", "best_score", "refused", "grade", "citation_correct", "answer"]]

**Read the failures** (grade ≠ correct, wrong refusals, wrong citations) and explain each one in the report.

In [ ]:
rag[(rag.grade != 'correct')][['qid', 'question', 'expected_answer', 'answer', 'retrieved']]

## 5. Fine-tuning: predict the stewards' outcome of an incident
The labelled incidents come from the stewards' decisions (`make prepare`): Fact + descriptive part of the Reason,
verdict removed, drivers anonymised. The test split is never used for training and its decisions are not in the
RAG index.

In [ ]:
!python scripts/train_classifier.py --curve 50,150

In [ ]:
import json
rep = json.load(open("eval/results/latest_classifier.json"))
pd.DataFrame([{"model": k, "test acc": v["test"]["accuracy"], "test macro-F1": v["test"]["macro_f1"],
               "hard acc": v["hard"]["accuracy"] if v["hard"] else None} for k, v in rep["results"].items()])

## 6. Figures for the report and the summary tables

In [ ]:
!pip -q install umap-learn matplotlib
!python scripts/make_figures.py
from IPython.display import Image, Markdown, display
for f in sorted(glob.glob("reports/figures/*.png")):
    display(Image(f, width=600))
display(Markdown(open("eval/results/results.md").read()))